# AgentSystems-02 — Perception active : la valeur d'information

**Série AgentSystems** · Prolongement d'AgentSystems-01 (*perception → action*)

**Navigation** : [← Retour à la série](README.md) · Précédent : [AgentSystems-01 — Perception → Action](AgentSystems-01-Perception-Vers-Action-Python.ipynb) · Jumeau Python : [AgentSystems-02 (Python)](AgentSystems-02-Perception-Active-Valeur-Information-Python.ipynb)

AgentSystems-01 a posé la boucle *percevoir → décider → agir*. Ce carnet ajoute la
couche qui fait de la perception un **choix** : l'agent ne lit pas tous les capteurs,
il observe **ce qui peut changer sa décision**. Chaque observation coûte ; la
**valeur d'information** d'une observation est la baisse du risque de décision qu'elle
rend possible. Ce carnet rend ce calcul exécutable : on mesure, capteur par capteur,
combien d'erreur il fait éviter, et on oppose la politique *active* (observer ce qui
compte) à une politique *naïve* (observer tout, dans l'ordre).

## Enjeu — de la perception passive à la perception active

Une perception **passive** lit tous les capteurs disponibles : elle est simple, mais
paie l'information qu'elle ne va pas utiliser. Une perception **active** raisonne
*avant* d'observer : « cette observation peut-elle changer ma décision ? ». Si oui, et
si la baisse d'erreur qu'elle promet vaut son coût, on l'achète ; sinon on s'abstient.

C'est le critère nominal de l'Epic #26930 : *choisir une information parce qu'elle
peut changer la décision*. Son instrument est la **valeur d'information** (value of
information) d'un échantillon : l'écart entre le risque bayésien avant et après
l'observation, moyenné sur tous les résultats possibles du capteur.

## 0. La décision sous incertitude

Un état latent (`bonne` / `mauvaise` campagne), deux actions (`lancer` / `ne pas
lancer`), une matrice de perte. La seule inconnue qui gouverne la décision est
`q = P(mauvais)`.

In [2]:
// La decision sous incertitude : un etat latent, deux actions, une matrice de perte.
const L_ACT: f64 = 100.0;   // perte si on lance une campagne mauvaise
const L_OPP: f64 = 30.0;    // perte d'opportunite si on ne lance pas une campagne bonne

fn bayes_risk(q: f64) -> f64 {
    // Risque bayesien : perte attendue de la meilleure decision possible quand P(mauvais)=q.
    let a = q * L_ACT;
    let b = (1.0 - q) * L_OPP;
    if a <= b { a } else { b }
}

fn optimal_action(q: f64) -> &'static str {
    // Action bayesienne : lancer si le risque de lancer est le plus bas.
    if q * L_ACT < (1.0 - q) * L_OPP { "lancer" } else { "ne pas lancer" }
}

let q_seuil: f64 = L_OPP / (L_ACT + L_OPP);  // on lance seulement si q est sous ce seuil
let q0: f64 = 0.40;                          // a priori, la campagne a 40 % de chances d'etre mauvaise

println!("Seuil bayesien q* = L_OPP/(L_ACT+L_OPP) = {:.3}", q_seuil);
println!("A priori q0 = {:.2}  ->  action optimale : {}", q0, optimal_action(q0));
println!("Risque bayesien a priori : {:.2}", bayes_risk(q0));
assert!(bayes_risk(q0) == if 0.40 * L_ACT <= 0.60 * L_OPP { 0.40 * L_ACT } else { 0.60 * L_OPP });  // = 18.0
assert!(optimal_action(q0) == "ne pas lancer");

Seuil bayesien q* = L_OPP/(L_ACT+L_OPP) = 0.231


A priori q0 = 0.40  ->  action optimale : ne pas lancer


Risque bayesien a priori : 18.00


### Lecture du résultat

Avec `q0 = 0.40`, au-dessus du seuil `0.231`, l'action bayésienne est **de ne pas
lancer** — et le risque bayésien (la perte attendue de la meilleure décision possible
avec l'information qu'on a) vaut **18**. Ce nombre est la référence : toute
observation qui le fait *baisser* porte de la valeur ; la question est de savoir de
combien, et au prix de quoi.

## 1. La valeur d'information d'un capteur

Un capteur est un modèle de bruit : `P(lecture | etat)`. Sa **valeur d'information**
est la baisse espérée du risque bayésien qu'il procure :

    EVOI(capteur) = RisqueBayesien(a priori) −  Σ_r  P(r) · RisqueBayesien(postérieur à r)

On ne la *cite* pas : on la **calcule** pour chaque capteur, en moyennant le risque
postérieur sur toutes les lectures que le capteur peut renvoyer. La valeur **nette**
soustrait le coût d'achat du capteur.

In [3]:
// Deux capteurs, meme cout d'achat, bruit different.
// P(lecture "mauvaise" | etat) : le capteur "test de marche" est nettement plus informatif.
const L_ACT: f64 = 100.0;   // perte si on lance une campagne mauvaise
const L_OPP: f64 = 30.0;    // perte d'opportunite si on ne lance pas une campagne bonne

fn bayes_risk(q: f64) -> f64 {
    // Risque bayesien : perte attendue de la meilleure decision possible quand P(mauvais)=q.
    let a = q * L_ACT;
    let b = (1.0 - q) * L_OPP;
    if a <= b { a } else { b }
}

struct Cap {
    // Un capteur est un modele de bruit : P(lecture | etat) et son cout d'achat.
    p_m_mauv: f64,  // P(lecture "mauvaise" | campagne mauvaise)
    p_m_bon: f64,   // P(lecture "mauvaise" | campagne bonne)
    cout: f64,      // cout d'achat du capteur
}

fn posterior_q(q: f64, cap: &Cap) -> (f64, f64, f64) {
    // Posterieur P(mauvais | lecture) pour les deux lectures, a partir du prior q.
    // Retourne (P(lecture="mauvaise"), P(mauvais|l="mauvaise"), P(mauvais|l="bonne")).
    let (a, b) = (cap.p_m_mauv, cap.p_m_bon);
    let p_l_m = a * q + b * (1.0 - q);            // P(lecture "mauvaise")
    let q_mauv = (a * q) / p_l_m;                  // P(mauvais | lecture "mauvaise")
    let q_bon = ((1.0 - a) * q) / (1.0 - p_l_m);   // P(mauvais | lecture "bonne")
    (p_l_m, q_mauv, q_bon)
}

fn evoi(q: f64, cap: &Cap) -> f64 {
    // Valeur d'information : baisse esperee du risque bayesien apportee par le capteur.
    let (p_l_m, q_mauv, q_bon) = posterior_q(q, cap);
    let risque_apres = p_l_m * bayes_risk(q_mauv) + (1.0 - p_l_m) * bayes_risk(q_bon);
    bayes_risk(q) - risque_apres
}

// Ordre d'insertion du dictionnaire Python : Vec de paires dans le meme ordre, jamais HashMap.
let capteurs: Vec<(&str, Cap)> = vec![
    ("test de marche",          Cap { p_m_mauv: 0.90, p_m_bon: 0.20, cout: 10.0 }),
    ("analyse concurrentielle", Cap { p_m_mauv: 0.70, p_m_bon: 0.45, cout: 10.0 }),
];
let q0: f64 = 0.40;  // a priori, la campagne a 40 % de chances d'etre mauvaise

println!("{:<24}{:>8}{:>7}{:>14}", "capteur", "EVOI", "cout", "valeur nette");
for (nom, cap) in &capteurs {
    let v = evoi(q0, cap);
    println!("{:<24}{:>8.2}{:>7.1}{:>14.2}", nom, v, cap.cout, v - cap.cout);
}
assert!(evoi(q0, &capteurs[0].1) > evoi(q0, &capteurs[1].1));

capteur                     EVOI   cout  valeur nette


test de marche             10.40   10.0          0.40


analyse concurrentielle     0.00   10.0        -10.00


### Lecture du résultat

Les deux capteurs coûtent le même prix, mais leur valeur n'est pas la même : le
**test de marché** baisse le risque de **10,40** points, tandis que l'**analyse
concurrentielle** en vaut **0** — son bruit est trop faible pour faire passer le
postérieur de l'autre côté du seuil de décision, elle ne change donc *aucune*
décision. C'est la leçon du tableau : **deux capteurs au même prix ne valent pas la
même information**, et un capteur peut en valoir *zéro*. La valeur *nette*
(EVOI − coût) décide de l'achat : le test de marché passe (+0,40), l'analyse
concurrentielle est un pur coût (−10,00).

## 2. La boucle de perception active

L'agent **actif** itère : (1) pour chaque capteur restant, calculer la valeur nette
au postérieur courant ; (2) acheter le capteur à la plus haute valeur nette *positive*
; (3) observer et mettre le postérieur à jour ; (4) répéter jusqu'à ce qu'aucun
capteur restant ait de valeur nette positive — alors on décide. Une politique
**naïve** achète, elle, tous les capteurs dans un ordre fixe, sans regarder leur
valeur. La trajectoire affichée est la réalisation la plus probable de chaque capteur
(la valeur d'information elle-même est déjà la moyenne sur toutes les lectures).

In [4]:
// La boucle de perception active opposee a la boucle naive.
const L_ACT: f64 = 100.0;   // perte si on lance une campagne mauvaise
const L_OPP: f64 = 30.0;    // perte d'opportunite si on ne lance pas une campagne bonne

fn bayes_risk(q: f64) -> f64 {
    // Risque bayesien : perte attendue de la meilleure decision possible quand P(mauvais)=q.
    let a = q * L_ACT;
    let b = (1.0 - q) * L_OPP;
    if a <= b { a } else { b }
}

fn optimal_action(q: f64) -> &'static str {
    // Action bayesienne : lancer si le risque de lancer est le plus bas.
    if q * L_ACT < (1.0 - q) * L_OPP { "lancer" } else { "ne pas lancer" }
}

#[derive(Clone)]
struct Cap {
    // Un capteur est un modele de bruit : P(lecture | etat) et son cout d'achat.
    p_m_mauv: f64,  // P(lecture "mauvaise" | campagne mauvaise)
    p_m_bon: f64,   // P(lecture "mauvaise" | campagne bonne)
    cout: f64,      // cout d'achat du capteur
}

fn posterior_q(q: f64, cap: &Cap) -> (f64, f64, f64) {
    // Posterieur P(mauvais | lecture) pour les deux lectures, a partir du prior q.
    let (a, b) = (cap.p_m_mauv, cap.p_m_bon);
    let p_l_m = a * q + b * (1.0 - q);
    let q_mauv = (a * q) / p_l_m;
    let q_bon = ((1.0 - a) * q) / (1.0 - p_l_m);
    (p_l_m, q_mauv, q_bon)
}

fn evoi(q: f64, cap: &Cap) -> f64 {
    // Valeur d'information : baisse esperee du risque bayesien apportee par le capteur.
    let (p_l_m, q_mauv, q_bon) = posterior_q(q, cap);
    let risque_apres = p_l_m * bayes_risk(q_mauv) + (1.0 - p_l_m) * bayes_risk(q_bon);
    bayes_risk(q) - risque_apres
}

fn percevoir_actif<'a>(
    mut q: f64,
    capteurs: &[(&'a str, Cap)],
    budget_obs: usize,
) -> (Vec<(&'a str, &'static str)>, f64, &'static str, f64) {
    // Perception active : n'acheter que les capteurs a valeur nette positive,
    // le plus valuable d'abord. Retourne (seq, q_fin, action, cout_total).
    let mut seq: Vec<(&'a str, &'static str)> = Vec::new();
    let mut cout = 0.0f64;
    let mut restants: Vec<(&'a str, Cap)> = capteurs.to_vec();
    for _ in 0..budget_obs {
        // max(net, key=net.get) : premier maximum dans l'ordre d'iteration -> comparaison stricte >
        let mut meilleur: Option<(usize, f64)> = None;
        for (i, (_, c)) in restants.iter().enumerate() {
            let n = evoi(q, c) - c.cout;
            if meilleur.is_none() || n > meilleur.unwrap().1 {
                meilleur = Some((i, n));
            }
        }
        let (idx, n) = meilleur.unwrap();
        if n <= 0.0 {
            break; // plus aucune observation ne vaut son prix
        }
        let (nom, cap) = &restants[idx];
        let (p_l_m, q_m, q_b) = posterior_q(q, cap);
        let lecture: &'static str = if p_l_m >= 0.5 { "mauvaise" } else { "bonne" };
        q = if lecture == "mauvaise" { q_m } else { q_b };
        seq.push((*nom, lecture));
        cout += cap.cout;
        restants.remove(idx);
    }
    (seq, q, optimal_action(q), cout)
}

fn percevoir_naif<'a>(
    mut q: f64,
    capteurs: &[(&'a str, Cap)],
    budget_obs: usize,
) -> (Vec<(&'a str, &'static str)>, f64, &'static str, f64) {
    // Perception naive : acheter tous les capteurs, dans l'ordre du vecteur.
    let mut seq: Vec<(&'a str, &'static str)> = Vec::new();
    let mut cout = 0.0f64;
    for (n, c) in capteurs.iter().take(budget_obs) {
        let (p_l_m, q_m, q_b) = posterior_q(q, c);
        let lecture: &'static str = if p_l_m >= 0.5 { "mauvaise" } else { "bonne" };
        q = if lecture == "mauvaise" { q_m } else { q_b };
        seq.push((*n, lecture));
        cout += c.cout;
    }
    (seq, q, optimal_action(q), cout)
}

// Ordre d'insertion du dictionnaire Python : Vec de paires dans le meme ordre.
let capteurs: Vec<(&str, Cap)> = vec![
    ("test de marche",          Cap { p_m_mauv: 0.90, p_m_bon: 0.20, cout: 10.0 }),
    ("analyse concurrentielle", Cap { p_m_mauv: 0.70, p_m_bon: 0.45, cout: 10.0 }),
];
let q0: f64 = 0.40;  // a priori, la campagne a 40 % de chances d'etre mauvaise

let (actif_seq, _, actif_action, actif_cout) = percevoir_actif(q0, &capteurs, 5);
let (naif_seq, _, naif_action, naif_cout) = percevoir_naif(q0, &capteurs, 5);
println!("POLITIQUE ACTIVE");
for (n, l) in &actif_seq {
    println!("  achete {}  (lecture: {})", n, l);
}
println!("  ->  decision: {}   |   cout total: {:.1}\n", actif_action, actif_cout);
println!("POLITIQUE NAIVE");
for (n, l) in &naif_seq {
    println!("  achete {}  (lecture: {})", n, l);
}
println!("  ->  decision: {}   |   cout total: {:.1}", naif_action, naif_cout);
assert!(actif_cout < naif_cout, "la politique active doit couter strictement moins cher");

POLITIQUE ACTIVE


  achete test de marche  (lecture: bonne)


  ->  decision: lancer   |   cout total: 10.0


POLITIQUE NAIVE


  achete test de marche  (lecture: bonne)


  achete analyse concurrentielle  (lecture: bonne)


  ->  decision: lancer   |   cout total: 20.0


### Le résultat — observer ce qui compte

La politique **active** s'arrête après le seul capteur qui vaille son prix ; la
politique **naïve** achète les deux. Même décision finale, **coût plus bas** pour
l'active : elle a refusé de payer l'information que le second capteur n'apportait
plus. Une fois le premier lu, le postérieur est déjà proche d'une certitude, et la
**valeur marginale** du second tombe sous son coût. Ce n'est pas l'information en
soi qui compte, mais l'information qui *change la décision*.

## Synthèse — la perception comme choix

| | Perception passive (naïve) | Perception active (valeur d'information) |
|---|---|---|
| Critère d'achat | tous les capteurs, à l'aveugle | seulement si `EVOI > coût` |
| Coût | somme de tous les prix | somme des prix *qui valaient le coup* |
| Décision | après tout | dès que plus rien ne vaut son prix |
| Lien avec l'Epic | « percevoir » | « **choisir** une information parce qu'elle peut changer la décision » |

La boucle d'AgentSystems-01 (percevoir → décider → agir) devient une boucle où la
perception est **argumentée** : chaque capteur justifie son prix par la baisse
d'erreur qu'il rend possible.

## Exercices

In [5]:
// Exercice 1 — Le troisieme capteur
// On ajoute un capteur "enquete client" : P(l="mauvaise"|mauv)=0.95,
// P(l="mauvaise"|bon)=0.05, cout 25. Calculer sa valeur d'information et sa valeur
// nette a l'a priori q0, et dire si la politique active l'acheterait (avant le
// test de marche, apres, ou jamais).
fn enquete_client() -> Option<f64> {
    // TODO etudiant
    None  // TODO etudiant
}

In [6]:
// Exercice 2 — Le seuil de decision
// A quel a priori q0 la decision bascule-t-elle de "ne pas lancer" a "lancer" ?
// Exprimer q* en fonction de L_ACT et L_OPP, puis le verifier par le code
// (le seuil doit satisfaire optimal_action(q* - e) = "ne pas lancer" et
// optimal_action(q* + e) = "lancer" pour un petit e).
fn seuil_decision() -> Option<f64> {
    // TODO etudiant
    None  // TODO etudiant
}

In [7]:
// Exercice 3 — La valeur d'information marginale
// Pourquoi la valeur du second capteur, calculee APRES le premier, est-elle plus
// faible que sa valeur calculee a priori ? Relier a la notion d'information marginale
// et a la condition d'arret de la boucle active (valeur nette <= 0).
fn valeur_marginale() -> Option<String> {
    // TODO etudiant
    None  // TODO etudiant
}

## À retenir

- La **valeur d'information** d'une observation est la baisse du risque bayésien qu'elle
  rend possible, moyennée sur ses résultats — on la *calcule*, on ne la cite pas.
- Une observation a une **valeur nette** positive seulement si `EVOI > coût` ; la
  perception active n'achète que celles-là.
- La **valeur marginale** d'un capteur chute une fois les autres lus : c'est ce qui
  fait s'arrêter la boucle active avant d'avoir tout observé.
- C'est le critère de l'Epic : *choisir une information parce qu'elle peut changer la
  décision* — la perception devient un argument, pas un réflexe.

## Bibliographie

- X. Chen, A. Guestrin, *Bayesian Decision Theory: A Perspective on the Uncertainty of
  Machine Learning*, 2019 — le risque bayésien comme objet de décision.
- I. Russo, *Entropy Inference: Common Information Measures for Continuous Variables*,
  2016 — la quantification de l'information.
- Carnet frère : **AgentSystems-01 — Perception vers Action** (la boucle percevoir →
  décider → agir posée en amont de ce carnet).